### batch2 of Birds

In [ ]:
import pandas as pd
import numpy as np



input_file = r"T:\Huihui\Bird\HVC12152025\Analysis\B3\Brain2_1\virus_celltype_summary2.xlsx"
output_dir = r"T:\Huihui\Bird\HVC12152025\Analysis\B3\Brain2_1"


df = pd.read_excel(input_file)

genes_for_classification = ["PLP1", "SOX9", "VGLUT2", "csf1r", "gad1"]

# Make sure these columns are numeric
df[genes_for_classification] = df[genes_for_classification].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)


df.loc[df["VGLUT2"] < 4, "VGLUT2"] = 0

df.loc[df["gad1"] < 2, "gad1"] = 0


df["max_gene"] = df[genes_for_classification].idxmax(axis=1)
df["max_count"] = df[genes_for_classification].max(axis=1)

df["n_genes_with_max"] = df[genes_for_classification].eq(
    df["max_count"], axis=0
).sum(axis=1)



def assign_cell_type(row):
    max_gene = row["max_gene"]
    max_count = row["max_count"]

    # If all selected marker counts are 0
    if max_count < 3:
        return "NA"

    # If there is a tie for the maximum marker
    # This avoids arbitrary classification when two markers have the same max count
    if row["n_genes_with_max"] > 1:
        return "unknown"

    if max_gene == "VGLUT2":
        # Avoid division by zero, although VGLUT2 should be > 0 here
        ratio = row["gad1"] / row["VGLUT2"]
        if ratio < 0.67:
            return "excitatory neuron"
        else:
            return "unknown"

    elif max_gene == "gad1":
        # Avoid division by zero, although gad1 should be > 0 here
        ratio = row["VGLUT2"] / row["gad1"]
        if ratio < 0.67:
            return "inhibitory neuron"
        else:
            return "unknown"

    elif max_gene == "PLP1":
        return "oligo"

    elif max_gene == "SOX9":
        return "astrocyte"

    elif max_gene == "csf1r":
        return "microglia"

    else:
        return "unknown"


df["cell_type"] = df.apply(assign_cell_type, axis=1)


celltype_counts_by_virus = pd.crosstab(
    df["virus"],
    df["cell_type"]
)

# Make sure all expected cell types are included as columns
expected_cell_types = [
    "excitatory neuron",
    "inhibitory neuron",
    "unknown",
    "oligo",
    "astrocyte",
    "microglia",
    "NA"
]

for cell_type in expected_cell_types:
    if cell_type not in celltype_counts_by_virus.columns:
        celltype_counts_by_virus[cell_type] = 0

celltype_counts_by_virus = celltype_counts_by_virus[expected_cell_types]


celltype_fraction_by_virus = celltype_counts_by_virus.div(
    celltype_counts_by_virus.sum(axis=1),
    axis=0
) * 100


print("Cell type counts by virus:")
print(celltype_counts_by_virus)

print("\nCell type percentage by virus:")
print(celltype_fraction_by_virus)

print("\nOverall cell type counts:")
print(df["cell_type"].value_counts())


df.to_excel(
    output_dir + r"\cells_with_assigned_cell_type.xlsx",
    index=False
)

celltype_counts_by_virus.to_excel(
    output_dir + r"\celltype_counts_by_virus.xlsx"
)

celltype_fraction_by_virus.to_excel(
    output_dir + r"\celltype_fraction_by_virus_percent.xlsx"
)

Cell type counts by virus:
cell_type  excitatory neuron  inhibitory neuron  unknown  oligo  astrocyte  \
virus                                                                        
EEEV                      11                  1        0      2          0   
SFV                       18                  0        0      0          0   
SINV                       5                  0        1      1          0   
VEEV                       3                  1        0      0          0   
WEEV                       3                  1        0      0          0   
no_gene3                   0                  1        0      0          0   

cell_type  microglia  NA  
virus                     
EEEV               0   1  
SFV                0   1  
SINV               0   0  
VEEV               0   0  
WEEV               0   0  
no_gene3           0   2  

Cell type percentage by virus:
cell_type  excitatory neuron  inhibitory neuron    unknown      oligo  \
virus                      

### batch1 of birds

In [ ]:
import pandas as pd
import numpy as np


input_file = r"T:\Huihui\Bird\Bird\Analysis\Bird5\Brain1_1\virus_celltype_summary2.xlsx"
output_dir = r"T:\Huihui\Bird\Bird\Analysis\Bird5\Brain1_1"


df = pd.read_excel(input_file)


genes_for_classification = ["PLP1", "SOX9", "VGLUT2", "VGLUT1","csf1r", "gad2"]

df[genes_for_classification] = df[genes_for_classification].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)


df.loc[df["VGLUT2"] < 3, "VGLUT2"] = 0
df.loc[df["VGLUT1"] < 3, "VGLUT1"] = 0

df.loc[df["gad2"] < 3, "gad2"] = 0
df.loc[df["PLP1"] < 3, "PLP1"] = 0
df.loc[df["SOX9"] < 2, "SOX9"] = 0


df["max_gene"] = df[genes_for_classification].idxmax(axis=1)
df["max_count"] = df[genes_for_classification].max(axis=1)

df["n_genes_with_max"] = df[genes_for_classification].eq(
    df["max_count"], axis=0
).sum(axis=1)


def assign_cell_type(row):
    max_gene = row["max_gene"]
    max_count = row["max_count"]

    # If all selected marker counts are 0
    if max_count < 2:
        return "NA"

    # If two or more genes have the same maximum count
    if row["n_genes_with_max"] > 1:
        return "unknown"

    if max_gene in ["VGLUT2", "VGLUT1"]:
        vglut_count = row[max_gene]
        ratio = row["gad2"] / vglut_count

        if ratio < 0.67:
            return "excitatory neuron"
        else:
            return "unknown"

    elif max_gene == "gad2":
        return "inhibitory neuron"

    elif max_gene == "PLP1":
        return "oligo"

    elif max_gene == "SOX9":
        return "astrocyte"

    elif max_gene == "csf1r":
        return "microglia"

    else:
        return "unknown"


df["cell_type"] = df.apply(assign_cell_type, axis=1)


celltype_counts_by_virus = pd.crosstab(
    df["virus"],
    df["cell_type"]
)

expected_cell_types = [
    "excitatory neuron",
    "inhibitory neuron",
    "unknown",
    "oligo",
    "astrocyte",
    "microglia"
]

for cell_type in expected_cell_types:
    if cell_type not in celltype_counts_by_virus.columns:
        celltype_counts_by_virus[cell_type] = 0

celltype_counts_by_virus = celltype_counts_by_virus[expected_cell_types]


celltype_fraction_by_virus = celltype_counts_by_virus.div(
    celltype_counts_by_virus.sum(axis=1),
    axis=0
) * 100


print("Cell type counts by virus:")
print(celltype_counts_by_virus)

print("\nCell type percentage by virus:")
print(celltype_fraction_by_virus)

print("\nOverall cell type counts:")
print(df["cell_type"].value_counts())


df.to_excel(
    output_dir + r"\cells_with_assigned_cell_type_gad2.xlsx",
    index=False
)

celltype_counts_by_virus.to_excel(
    output_dir + r"\celltype_counts_by_virus_gad2.xlsx"
)

celltype_fraction_by_virus.to_excel(
    output_dir + r"\celltype_fraction_by_virus_percent_gad2.xlsx"
)

Cell type counts by virus:
cell_type  excitatory neuron  inhibitory neuron  unknown  oligo  astrocyte  \
virus                                                                        
EEEV                      54                  2        4      5          2   
SFV                       23                  1        1      6          3   
SINV                       4                  1        0      1          0   
VEEV                       2                  0        0      0          0   
WEEV                      14                  2        0      1          3   
no_gene4                   0                  1        0      0          0   

cell_type  microglia  
virus                 
EEEV               0  
SFV                0  
SINV               0  
VEEV               0  
WEEV               0  
no_gene4           1  

Cell type percentage by virus:
cell_type  excitatory neuron  inhibitory neuron   unknown      oligo  \
virus                                                       